In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns',None)

In [2]:
df = pd.read_csv("../data/processed/loan_data_cleaned.csv")
print(df.shape)

(1345350, 44)


In [3]:
df['installment_to_income'] = df['installment']/(df['annual_inc']/12)

df['installment_to_income'] = df['installment_to_income'].replace([np.inf, -np.inf],np.nan)

# Clipping the WHOLE column at 99th percentile - this catches both inf-turned-NaN 
# AND the near-zero-income extreme finite values in one step
cap_value = df['installment_to_income'].quantile(0.99)
df['installment_to_income'] = df['installment_to_income'].clip(upper=cap_value)

df['installment_to_income'] = df['installment_to_income'].fillna(cap_value)

print(df['installment_to_income'].describe())

count    1.345350e+06
mean     7.890733e-02
std      4.238298e-02
min      5.916000e-04
25%      4.649550e-02
50%      7.221600e-02
75%      1.054349e-01
max      1.993551e-01
Name: installment_to_income, dtype: float64


In [4]:
print((df['annual_inc'] == 0).sum())
print(df[df['annual_inc'] == 0]['installment_to_income'])

361
16142      0.199355
57633      0.199355
375767     0.199355
375880     0.199355
376271     0.199355
             ...   
1321496    0.199355
1323189    0.199355
1326690    0.199355
1326698    0.199355
1329239    0.199355
Name: installment_to_income, Length: 361, dtype: float64


In [5]:
df['loan_to_income'] = df['loan_amnt'] / df['annual_inc']

df['loan_to_income'] = df['loan_to_income'].replace([np.inf, -np.inf], np.nan)

cap_value_lti = df['loan_to_income'].quantile(0.99)
df['loan_to_income'] = df['loan_to_income'].clip(upper=cap_value_lti)
df['loan_to_income'] = df['loan_to_income'].fillna(cap_value_lti)

print(df['loan_to_income'].describe())

count    1.345350e+06
mean     2.136561e-01
std      1.135482e-01
min      4.000000e-03
25%      1.250000e-01
50%      2.000000e-01
75%      2.909091e-01
max      5.000000e-01
Name: loan_to_income, dtype: float64


In [6]:
print(df['earliest_cr_line'].dtype)
print(df['earliest_cr_line'].head(10))

str
0    Aug-2003
1    Dec-1999
2    Aug-2000
3    Jun-1998
4    Oct-1987
5    Jun-1990
6    Feb-1999
7    Apr-2002
8    Nov-1994
9    Jun-1996
Name: earliest_cr_line, dtype: str


In [7]:
print('issue_d' in df.columns)

False


In [8]:
issue_d_full = pd.read_csv('../data/raw/accepted_2007_to_2018Q4.csv', usecols=['issue_d'])
print(issue_d_full.shape)
print(df.index.max())

(2260701, 1)
1345349


In [9]:
df['issue_d'] = issue_d_full.loc[df.index, 'issue_d']
print(df['issue_d'].head(10))
print(df['issue_d'].isnull().sum())

0    Dec-2015
1    Dec-2015
2    Dec-2015
3    Dec-2015
4    Dec-2015
5    Dec-2015
6    Dec-2015
7    Dec-2015
8    Dec-2015
9    Dec-2015
Name: issue_d, dtype: str
14


In [10]:
df['earliest_cr_line_dt'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')
df['issue_d_dt'] = pd.to_datetime(df['issue_d'], format='%b-%Y')

print(df[['earliest_cr_line', 'earliest_cr_line_dt', 'issue_d', 'issue_d_dt']].head(10))

  earliest_cr_line earliest_cr_line_dt   issue_d issue_d_dt
0         Aug-2003          2003-08-01  Dec-2015 2015-12-01
1         Dec-1999          1999-12-01  Dec-2015 2015-12-01
2         Aug-2000          2000-08-01  Dec-2015 2015-12-01
3         Jun-1998          1998-06-01  Dec-2015 2015-12-01
4         Oct-1987          1987-10-01  Dec-2015 2015-12-01
5         Jun-1990          1990-06-01  Dec-2015 2015-12-01
6         Feb-1999          1999-02-01  Dec-2015 2015-12-01
7         Apr-2002          2002-04-01  Dec-2015 2015-12-01
8         Nov-1994          1994-11-01  Dec-2015 2015-12-01
9         Jun-1996          1996-06-01  Dec-2015 2015-12-01


In [11]:
df['credit_history_years'] = (df['issue_d_dt'] - df['earliest_cr_line_dt']).dt.days / 365.25
print(df['credit_history_years'].describe())

count    1.345336e+06
mean     1.711273e+01
std      7.754718e+00
min     -4.188912e-01
25%      1.183025e+01
50%      1.583299e+01
75%      2.116085e+01
max      8.391513e+01
Name: credit_history_years, dtype: float64


In [12]:
print((df['credit_history_years'] < 0).sum())
print(df['credit_history_years'].isnull().sum())

267
14


In [13]:
df['credit_history_years'] = df['credit_history_years'].clip(lower=0)

print((df['credit_history_years'] < 0).sum())
print(df['credit_history_years'].describe())

0
count    1.345336e+06
mean     1.711277e+01
std      7.754623e+00
min      0.000000e+00
25%      1.183025e+01
50%      1.583299e+01
75%      2.116085e+01
max      8.391513e+01
Name: credit_history_years, dtype: float64


In [14]:
median_history = df['credit_history_years'].median()
df['credit_history_years'] = df['credit_history_years'].fillna(median_history)

print(df['credit_history_years'].isnull().sum())

0


In [15]:
df = df.drop(columns=['earliest_cr_line_dt', 'issue_d_dt'])
print(df.shape)
print(df.columns.tolist())

(1345350, 48)
['loan_amnt', 'term', 'installment', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'loan_status', 'purpose', 'addr_state', 'dti', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'application_type', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'mo_sin_old_rev_tl_op', 'mort_acc', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_tl_op_past_12m', 'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec_bankruptcies', 'tax_liens', 'total_bc_limit', 'total_il_high_credit_limit', 'default', 'never_delinq', 'never_public_record', 'installment_to_income', 'loan_to_income', 'issue_d', 'credit_history_years']


In [16]:
df = df.drop(columns=['issue_d'])

In [17]:
print(df.shape)
print(df.columns.tolist())

(1345350, 47)
['loan_amnt', 'term', 'installment', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'loan_status', 'purpose', 'addr_state', 'dti', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'application_type', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'mo_sin_old_rev_tl_op', 'mort_acc', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_tl_op_past_12m', 'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec_bankruptcies', 'tax_liens', 'total_bc_limit', 'total_il_high_credit_limit', 'default', 'never_delinq', 'never_public_record', 'installment_to_income', 'loan_to_income', 'credit_history_years']


In [18]:
df['revol_bal_to_income'] = df['revol_bal'] / df['annual_inc']

df['revol_bal_to_income'] = df['revol_bal_to_income'].replace([np.inf, -np.inf], np.nan)
cap_value_rbi = df['revol_bal_to_income'].quantile(0.99)
df['revol_bal_to_income'] = df['revol_bal_to_income'].clip(upper=cap_value_rbi)
df['revol_bal_to_income'] = df['revol_bal_to_income'].fillna(cap_value_rbi)

print(df['revol_bal_to_income'].describe())

count    1.345350e+06
mean     2.218225e-01
std      1.752435e-01
min      0.000000e+00
25%      9.866309e-02
50%      1.798000e-01
75%      2.959394e-01
max      9.571979e-01
Name: revol_bal_to_income, dtype: float64


In [19]:
df['total_credit_util'] = df['tot_cur_bal'] / (df['total_bc_limit'] + df['total_il_high_credit_limit'])

df['total_credit_util'] = df['total_credit_util'].replace([np.inf, -np.inf], np.nan)
cap_value_tcu = df['total_credit_util'].quantile(0.99)
df['total_credit_util'] = df['total_credit_util'].clip(upper=cap_value_tcu)
df['total_credit_util'] = df['total_credit_util'].fillna(cap_value_tcu)

print(df['total_credit_util'].describe())

count    1.345350e+06
mean     2.693610e+00
std      3.661613e+00
min      0.000000e+00
25%      7.593726e-01
50%      1.180941e+00
75%      3.199406e+00
max      2.333303e+01
Name: total_credit_util, dtype: float64


In [20]:
df = df.drop(columns=['total_credit_util'])
print(df.shape)

(1345350, 48)


In [21]:
df.to_csv('../data/processed/loan_data_features_wip.csv', index=False)
print(f"Saved {df.shape[0]} rows, {df.shape[1]} columns")

Saved 1345350 rows, 48 columns


In [2]:
df = pd.read_csv('../data/processed/loan_data_features_wip.csv')

In [3]:
print(df.shape)
print(df.columns.tolist())

(1345350, 48)
['loan_amnt', 'term', 'installment', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'loan_status', 'purpose', 'addr_state', 'dti', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'application_type', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'mo_sin_old_rev_tl_op', 'mort_acc', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_tl_op_past_12m', 'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec_bankruptcies', 'tax_liens', 'total_bc_limit', 'total_il_high_credit_limit', 'default', 'never_delinq', 'never_public_record', 'installment_to_income', 'loan_to_income', 'credit_history_years', 'revol_bal_to_income']


In [4]:
print(df['bc_util'].describe())

count    1.345350e+06
mean     6.008931e+01
std      2.764293e+01
min      0.000000e+00
25%      3.960000e+01
50%      6.320000e+01
75%      8.390000e+01
max      3.396000e+02
Name: bc_util, dtype: float64


In [5]:
print(df['bc_util'].quantile([0.75, 0.85, 0.90, 0.95]))
print((df['bc_util'] > 90).sum())
print((df['bc_util'] > 100).sum())

0.75    83.9
0.85    91.8
0.90    95.1
0.95    98.0
Name: bc_util, dtype: float64
232664
21261


In [6]:
df['high_utilization_flag'] = (df['bc_util'] > 90).astype(int)

print(df['high_utilization_flag'].value_counts())
print(df['high_utilization_flag'].value_counts(normalize=True))

high_utilization_flag
0    1112686
1     232664
Name: count, dtype: int64
high_utilization_flag
0    0.827061
1    0.172939
Name: proportion, dtype: float64


In [8]:
print(df['num_tl_op_past_12m'].describe())

count    1.345350e+06
mean     2.169913e+00
std      1.798573e+00
min      0.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      3.000000e+00
max      3.200000e+01
Name: num_tl_op_past_12m, dtype: float64


In [9]:
print(df['inq_last_6mths'].describe())

count    1.345350e+06
mean     6.550756e-01
std      9.377688e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+00
max      8.000000e+00
Name: inq_last_6mths, dtype: float64


In [11]:
print((df['num_tl_op_past_12m'] >= 4).sum())
print((df['inq_last_6mths'] >= 2).sum())

248460
208215


In [12]:
combined_flag_test = ((df['num_tl_op_past_12m'] >= 4) | (df['inq_last_6mths'] >= 2))
print(combined_flag_test.sum())
print(combined_flag_test.mean())

381402
0.2834964879027762


In [13]:
df.to_csv('../data/processed/loan_data_features_wip.csv', index=False)
print(f"Saved {df.shape[0]} rows, {df.shape[1]} columns")

Saved 1345350 rows, 49 columns
